# 02 · ¿En qué porcentaje de tiendas se rompe la escalera?

**Objetivo:** Mostrar a revenue management y category management cómo se comporta en el anaquel la arquitectura de precios por tamaño de marcas líderes, y dónde se rompe.

**Pregunta:** ¿Qué parte de los escalones tiene el tamaño grande más caro por unidad, por marca y por escalón?

Importa porque mide qué tan seguido el cliente que compra más paga más por unidad. IC90 = rango probable
(bootstrap de escalones y ciudades).

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import escalera as S
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pts, st, cat = S.load()
REPS = 500
p = pd.read_csv(PROC / "escalones.csv")
SEL = json.loads((META / "parametros.json").read_text(encoding="utf-8"))["seleccionadas"]
tot = S.broken(p, reps=REPS)
print({k: round(v, 1) if isinstance(v, float) else v for k, v in tot.items()})
bm = S.by(p, "escalera", reps=REPS).set_index("escalera").loc[SEL]
display(bm.round(1))
be = S.by(p[p.contiguo], ["escalera", "escalon"], reps=REPS)
display(be.round(1))

{'escalones': 2138, 'tiendas': 300, 'municipios': 63, 'pct_roto': np.float64(17.4), 'pct_plano': np.float64(8.6), 'cambio_mediano': -13.9, 'cambio_roto_mediano': 3.7, 'ic90_inf': np.float64(4.2), 'ic90_sup': np.float64(30.5)}


,escalones,tiendas,municipios,pct_roto,pct_plano,cambio_mediano,cambio_roto_mediano,ic90_inf,ic90_sup
escalera,,,,,,,,,
Coca-Cola,611,266,60,0.8,7.7,-43.3,5.0,0.0,3.0
Huevo San Juan,399,223,56,43.9,8.5,-3.1,2.5,1.0,85.4
Huevo El Calvario,262,141,38,40.8,15.3,-2.4,5.4,6.2,73.9
Zucaritas,397,255,61,6.0,6.8,-25.3,4.8,0.0,11.9
Chiles La Costeña en rajas,469,273,62,12.8,7.5,-13.7,5.9,0.0,33.7


,escalera,escalon,escalones,tiendas,municipios,pct_roto,pct_plano,cambio_mediano,cambio_roto_mediano,ic90_inf,ic90_sup
0,Chiles La Costeña en rajas,105 g → 220 g,76,76,47,0.0,2.6,-9.3,NaN,0.0,0.0
1,Chiles La Costeña en rajas,240 g → 340 g,120,120,55,45.8,7.5,-2.8,5.9,37.2,54.4
2,Coca-Cola,1 L → 2.75 L,150,150,39,0.0,0.0,-45.7,NaN,0.0,0.0
3,Coca-Cola,355 ml → 600 ml,247,247,58,0.0,0.0,-44.5,NaN,0.0,0.0
4,Coca-Cola,600 ml → 1 L,199,199,43,2.5,23.6,-13.9,5.0,0.0,4.7
5,Huevo El Calvario,12 pzas → 18 pzas,126,126,35,10.3,15.1,-8.6,18.8,5.9,15.2
6,Huevo El Calvario,18 pzas → 30 pzas,128,128,37,73.4,13.3,2.6,5.4,68.3,79.4
7,Huevo San Juan,12 pzas → 18 pzas,199,199,55,85.9,11.6,2.4,2.5,82.3,90.1
8,Huevo San Juan,18 pzas → 30 pzas,191,191,54,1.6,5.8,-9.3,12.9,0.5,3.3
9,Zucaritas,30 g → 380 g,143,143,54,0.0,0.0,-51.8,NaN,0.0,0.0


## Sensibilidad

- solo escalones contiguos;
- sin escalones con algún precio extremo;
- ruptura solo si el grande cuesta más de 1% más por unidad (deja fuera redondeos de centavos).

In [2]:
var = {"Principal": p, "Solo contiguos": p[p.contiguo], "Sin extremos": p[~p.extremo]}
sens = pd.DataFrame({k: S.by(v, "escalera", reps=200).set_index("escalera").pct_roto for k, v in var.items()})
sens["Ruptura > 1%"] = p.groupby("escalera").cambio_pct.apply(lambda v: (v > 1).mean() * 100)
sens.loc["Total"] = [S.broken(v, reps=10)["pct_roto"] for v in var.values()] + [(p.cambio_pct > 1).mean() * 100]
display(sens.round(1))

,Principal,Solo contiguos,Sin extremos,Ruptura > 1%
escalera,,,,
Chiles La Costeña en rajas,12.8,27.5,12.8,12.2
Coca-Cola,0.8,0.8,0.8,0.5
Huevo El Calvario,40.8,42.1,40.8,39.7
Huevo San Juan,43.9,44.6,44.0,42.1
Zucaritas,6.0,6.0,6.1,5.8
Total,17.4,20.8,17.4,16.6


In [3]:
bm.to_csv(PROC / "rupturas_por_marca.csv", encoding="utf-8-sig")
be.to_csv(PROC / "rupturas_por_escalon.csv", index=False, encoding="utf-8-sig")
sens.to_csv(PROC / "sensibilidad_rupturas.csv", encoding="utf-8-sig")
F = "02_rupturas"
reg("roto_total", tot["pct_roto"], [tot["ic90_inf"], tot["ic90_sup"]], n=tot["escalones"], fuente=F)
reg("plano_total", tot["pct_plano"], n=tot["escalones"], fuente=F)
for k, r_ in bm.iterrows():
    kk = S.KEY[k]
    reg(f"roto_{kk}", r_.pct_roto, [r_.ic90_inf, r_.ic90_sup], n=r_.escalones, fuente=F)
    reg(f"plano_{kk}", r_.pct_plano, fuente=F); reg(f"tiendas_{kk}", r_.tiendas, fuente=F)
E_ = be.set_index(["escalera", "escalon"])
for (k, e), kk in {("Huevo San Juan", "12 pzas → 18 pzas"): "sj_12_18", ("Huevo El Calvario", "18 pzas → 30 pzas"): "ec_18_30",
                   ("Chiles La Costeña en rajas", "240 g → 340 g"): "raj_240_340", ("Zucaritas", "380 g → 600 g"): "zuc_380_600",
                   ("Huevo San Juan", "18 pzas → 30 pzas"): "sj_18_30", ("Huevo El Calvario", "12 pzas → 18 pzas"): "ec_12_18"}.items():
    r_ = E_.loc[(k, e)]
    reg(f"roto_{kk}", r_.pct_roto, [r_.ic90_inf, r_.ic90_sup], n=r_.escalones, fuente=F)
reg("sens_total_min", sens.loc["Total"].min(), fuente=F); reg("sens_total_max", sens.loc["Total"].max(), fuente=F)
reg("sens_sj_min", sens.loc["Huevo San Juan"].min(), fuente=F); reg("sens_sj_max", sens.loc["Huevo San Juan"].max(), fuente=F)
display(Markdown(
    f"**Conclusión.** De {tot['escalones']:,} escalones, **{tot['pct_roto']:.0f}%** están rotos (IC90 {tot['ic90_inf']:.0f}–{tot['ic90_sup']:.0f}%) "
    f"y {tot['pct_plano']:.0f}% son planos. La ruptura se concentra en un escalón por marca: huevo San Juan de 12 a 18 piezas "
    f"({E_.loc[('Huevo San Juan', '12 pzas → 18 pzas'), 'pct_roto']:.0f}% de las tiendas), El Calvario de 18 a 30 "
    f"({E_.loc[('Huevo El Calvario', '18 pzas → 30 pzas'), 'pct_roto']:.0f}%) y rajas La Costeña de 240 a 340 g "
    f"({E_.loc[('Chiles La Costeña en rajas', '240 g → 340 g'), 'pct_roto']:.0f}%). Coca-Cola casi nunca: {bm.loc['Coca-Cola', 'pct_roto']:.1f}%."))

**Conclusión.** De 2,138 escalones, **17%** están rotos (IC90 4–31%) y 9% son planos. La ruptura se concentra en un escalón por marca: huevo San Juan de 12 a 18 piezas (86% de las tiendas), El Calvario de 18 a 30 (73%) y rajas La Costeña de 240 a 340 g (46%). Coca-Cola casi nunca: 0.8%.